In [1]:
import time, requests
import numpy as np
import chromadb

OLLAMA = "http://localhost:11434"
MODEL = "qwen3-embedding:4b"
BOYUT = 1024
QWEN_TALIMAT = ("Instruct: Given a question about a video game, "
                "retrieve player reviews that answer the question\nQuery: ")

def sorgu_vektoru(soru, talimat=QWEN_TALIMAT):
    r = requests.post(f"{OLLAMA}/api/embed",
                      json={"model": MODEL, "input": talimat + soru}, timeout=60)
    r.raise_for_status()
    v = np.array(r.json()["embeddings"][0][:BOYUT], dtype=np.float32)
    return v / np.linalg.norm(v)

istemci = chromadb.PersistentClient(path="../data/chroma_v2")
yorumlar = istemci.get_collection("yorumlar")
oyunlar = istemci.get_collection("oyunlar")
print("yorumlar:", yorumlar.count(), "| oyunlar:", oyunlar.count())

t = time.time()
v = sorgu_vektoru("Gece oynamak korkutucu mu?")
sonuc = yorumlar.query(query_embeddings=[v.tolist()], n_results=5,
                       where={"app_id": 239140},
                       include=["documents", "metadatas", "distances"])
print(f"süre: {time.time() - t:.1f} sn\n")

for metin, meta, mesafe in zip(sonuc["documents"][0], sonuc["metadatas"][0], sonuc["distances"][0]):
    print(f"{mesafe:.3f} | olumlu={meta['olumlu']} | dil={meta['dil']} | faydalı={meta['faydali_oy']} | {meta['tarih']}")
    print(metin)
    print("-" * 80)

for m in requests.get(f"{OLLAMA}/api/ps").json()["models"]:
    print(m["name"], "| size:", m["size"], "| size_vram:", m["size_vram"])

yorumlar: 2644490 | oyunlar: 23218
süre: 19.0 sn

0.329 | olumlu=1 | dil=zh | faydalı=837 | 2022-10-25
晚上比早上安全得多，晚上是比较适合新手探索。而且没有任何恐怖元素，胆小的人可以尝试。夜魔名字听起来可怕，其实夜魔挺温和的而且经验很多，不要被其他评论影响了。这游戏也很适合儿童游玩
--------------------------------------------------------------------------------
0.332 | olumlu=1 | dil=en | faydalı=684 | 2015-01-31
"The Walking Dead" by day, "I Am Legend" by night;I can't tell you how beautiful this game looks without you just seeing it for yourself,  but I will say this.This is the first game, in a very long time, which has legitamately inspired a sense of fear in me.At night time, this game becomes a heart pounding nightmare.Like horror? Play this game.
--------------------------------------------------------------------------------
0.349 | olumlu=1 | dil=zh | faydalı=1 | 2018-10-30
好玩，就是有点难，夜魔一直让人慌，妈蛋，打到旧城区还是怕夜魔
--------------------------------------------------------------------------------
0.401 | olumlu=1 | dil=en | faydalı=1887 | 2015-01-28
Glad to see the misinformed 

## Adım 1 · Chroma bağlantısı ve ilk arama
- `data/chroma_v2` açıldı: yorumlar 2.644.490 parça, oyunlar 23.218 (05 ile aynı ✓).
- Sorgu vektörü: Qwen3-embedding:4b (Ollama), QWEN_TALIMAT öneki + ilk 1024 boyut + normalize. Yorumlar 05'te aynı yoldan embed edildi.
- Qwen GPU'da (`/api/ps`: size = size_vram = 3,43 GB).
- İlk sorgu 19 sn (HNSW indeksi RAM'e yükleniyor), sonrakiler hızlı.
- Test: Dying Light (239140) "Gece oynamak korkutucu mu?" → 5 sonuç da gece/korku hakkında, mesafe 0,329–0,404.
- **Sorun 1, tekrar:** aynı yorumun 2 parçası 5 yerin 2'sini aldı → sonuçları review_id ile tekilleştir.
- **Sorun 2, şaka yorum:** 1. sonuç (zh, 837 faydalı oy) "hiç korku yok, çocuklar için uygun" diyor. İronik. Faydalı oy ciddiyet garantisi değil → özetle modunda ve agent kurallarında ele alınacak.

In [2]:
def filtre_kur(app_id=None, olumlu=None, dil=None):
    kosullar = []
    if app_id is not None:
        kosullar.append({"app_id": app_id})
    if olumlu is not None:
        kosullar.append({"olumlu": olumlu})
    if dil is not None:
        kosullar.append({"dil": dil})
    if len(kosullar) == 0:
        return None
    if len(kosullar) == 1:
        return kosullar[0]
    return {"$and": kosullar}

def konu_ara(soru, app_id=None, olumlu=None, dil=None, k=5):
    v = sorgu_vektoru(soru)
    sonuc = yorumlar.query(query_embeddings=[v.tolist()], n_results=k * 4,
                           where=filtre_kur(app_id, olumlu, dil),
                           include=["documents", "metadatas", "distances"])
    gorulen = set()
    liste = []
    for metin, meta, mesafe in zip(sonuc["documents"][0], sonuc["metadatas"][0], sonuc["distances"][0]):
        if meta["review_id"] in gorulen:
            continue
        gorulen.add(meta["review_id"])
        liste.append({"mesafe": round(mesafe, 3), **meta, "metin": metin})
        if len(liste) == k:
            break
    return liste

def yazdir(liste):
    for s in liste:
        print(f"{s['mesafe']} | review_id={s['review_id']} | parça {s['parca_no']}/{s['parca_sayisi']} | "
              f"olumlu={s['olumlu']} | dil={s['dil']} | faydalı={s['faydali_oy']} | {s['tarih']}")
        print(s["metin"])
        print("-" * 80)

t = time.time()
print("=== Test 1: Dying Light, olumsuz, kasma ===")
yazdir(konu_ara("Oyun kasıyor mu, FPS düşüyor mu?", app_id=239140, olumlu=0))
print(f"süre: {time.time() - t:.1f} sn\n")

t = time.time()
print("=== Test 2: Dying Light, Türkçe yorumlar ===")
yazdir(konu_ara("Oyun kasıyor mu, FPS düşüyor mu?", app_id=239140, dil="tr", k=3))
print(f"süre: {time.time() - t:.1f} sn")

=== Test 1: Dying Light, olumsuz, kasma ===
0.405 | review_id=239140_111398 | parça 0/1 | olumlu=0 | dil=es | faydalı=0 | 2023-11-17
Si pudiera poner un "Meh", lo pondría.Es un juego repetitivo, pero eso no quita que es divertido, y la mecánica que lo separa de los demás juegos de zombies, el parkour, es bastante fluido aunque de vez en cuando te encontraras con una estructura que no puedes escalar porque al juego se le antoja no dejarte, o no detecta el input.Lo que más me causo problemas es el aspecto técnico del juego. Tengo una PC que supera por mucho los specs recomendados para jugar el juego y aún así le cuesta trabajo mantener los FPS a 60 o más y constantemente tienes bajones a los 20 e incluso a los 10 FPS sin importar la calidad gráfica o cuantas opciones desactives para quitar carga del GPU.Si lo encuentras en oferta lo puedo recomendar, pero no pagues precio completo por este juego ni sus DLCs
--------------------------------------------------------------------------------


## Adım 2 · konu_ara: tekilleştirme ve filtreler
- `filtre_kur`: app_id / olumlu / dil → Chroma `where` (tek koşul düz, çoklu `$and`). `olumlu=0` için `is not None` kontrolü şart.
- `konu_ara`: k×4 parça çekilir, review_id ile tekilleştirilir (ilk görülen = en yakın parça), sözlük listesi döner.
- Filtreler doğru: Test 1'de 5/5 olumsuz, Test 2'de 3/3 Türkçe. Tekrar yok ✓.
- **Kalite:** Dying Light olumsuz + "kasma" sorgusunda sadece 1. sonuç (es, 0,405, "10–20 FPS'e düşüyor") gerçekten ilgili, diğerleri genel şikâyet (0,47–0,57).
  - Sebep: DL %95 olumlu → 500 yorumluk karma seçimde ~25 olumsuz yorum var. Filtre havuzu daraltıyor, Chroma alakasız olsa da 5 sonuç getiriyor.
  - Ders: mesafe güven sinyali. Yüksek mesafeli sonuçlar agent'a "zayıf eşleşme" diye işaretlenmeli.
- Türkçe sonuçlar Steam "✅ şablon" yorumları (Buglar: Karşılaşmadım, PC: Ortalama). Kısmen bilgi veriyor.
- **Hız sorunu:** filtreli sorgu 84,5 sn ve 21,4 sn (indeks zaten yüklüydü). Bir sonraki adımda ölçülecek.

In [3]:
v = sorgu_vektoru("Oyun kasıyor mu, FPS düşüyor mu?")

t = time.time()
sorgu_vektoru("Oyun kasıyor mu, FPS düşüyor mu?")
print(f"embedding (Ollama): {time.time() - t:.2f} sn\n")

denemeler = {
    "filtresiz":       None,
    "app_id":          {"app_id": 239140},
    "app_id + olumlu": {"$and": [{"app_id": 239140}, {"olumlu": 0}]},
    "app_id + dil":    {"$and": [{"app_id": 239140}, {"dil": "tr"}]},
}

for ad, filtre in denemeler.items():
    for tur in [1, 2]:
        t = time.time()
        yorumlar.query(query_embeddings=[v.tolist()], n_results=20, where=filtre,
                       include=["documents", "metadatas", "distances"])
        print(f"{ad:16} | tur {tur} | {time.time() - t:6.2f} sn")

print("\n--- Alternatif: oyunun parçalarını çek, benzerliği numpy ile hesapla ---")
for tur in [1, 2]:
    t = time.time()
    hepsi = yorumlar.get(where={"app_id": 239140},
                         include=["embeddings", "documents", "metadatas"])
    t_get = time.time() - t
    benzerlik = np.array(hepsi["embeddings"]) @ v
    sira = np.argsort(-benzerlik)[:20]
    print(f"tur {tur} | get: {t_get:.2f} sn | numpy: {time.time() - t - t_get:.4f} sn | "
          f"parça: {len(hepsi['ids'])} | en yakın mesafe: {1 - benzerlik[sira[0]]:.3f}")

embedding (Ollama): 0.04 sn

filtresiz        | tur 1 |   0.08 sn
filtresiz        | tur 2 |   0.00 sn
app_id           | tur 1 |  16.76 sn
app_id           | tur 2 |   2.39 sn
app_id + olumlu  | tur 1 |  64.27 sn
app_id + olumlu  | tur 2 |  72.46 sn
app_id + dil     | tur 1 |  16.80 sn
app_id + dil     | tur 2 |   4.02 sn

--- Alternatif: oyunun parçalarını çek, benzerliği numpy ile hesapla ---
tur 1 | get: 0.33 sn | numpy: 0.0387 sn | parça: 238 | en yakın mesafe: 0.404
tur 2 | get: 0.27 sn | numpy: 0.0003 sn | parça: 238 | en yakın mesafe: 0.404


## Adım 2b · Hız ölçümü
| Sorgu | Tur 1 | Tur 2 |
|---|---|---|
| Embedding (Ollama) | 0,04 sn | |
| Filtresiz (HNSW, 2,6 M parça) | 0,08 sn | 0,00 sn |
| where app_id | 16,8 sn | 2,4 sn |
| where app_id + olumlu | 64,3 sn | 72,5 sn |
| where app_id + dil | 16,8 sn | 4,0 sn |
| **get(app_id) + numpy** | **0,33 sn** | **0,27 sn** |

- Yavaşlatan şey HNSW değil, Chroma'nın metadata filtresi (SQLite'ta 2,6 M kayıt). `olumlu=0` çok kayda uyduğu için kesişim en yavaşı ve önbellek de yardım etmiyor.
- Alternatif: oyunun tüm parçalarını vektörleriyle `get` ile çek (DL: 238 parça = 200 yorum + parçalar), `vektörler @ soru` ile cosine benzerlik, olumlu/dil filtresi Python'da.
- Aynı sonuç: en yakın mesafe 0,404 (Chroma 0,405, yuvarlama farkı).
- **Karar:** oyun belirtilmişse get + numpy. Oyun belirtilmemişse filtresiz HNSW.
- Bonus: "özetle" ve "hepsini getir" de aynı "oyunun parçalarını çek" adımını kullanabilir.

In [4]:
def oyun_parcalari(app_id):
    return yorumlar.get(where={"app_id": app_id},
                        include=["embeddings", "documents", "metadatas"])

def konu_ara(soru, app_id=None, olumlu=None, dil=None, k=5):
    v = sorgu_vektoru(soru)
    if app_id is not None:
        h = oyun_parcalari(app_id)
        if len(h["ids"]) == 0:
            return []
        mesafeler = 1 - np.array(h["embeddings"]) @ v
        adaylar = sorted(zip(h["documents"], h["metadatas"], mesafeler), key=lambda x: x[2])
    else:
        s = yorumlar.query(query_embeddings=[v.tolist()], n_results=k * 20,
                           include=["documents", "metadatas", "distances"])
        adaylar = zip(s["documents"][0], s["metadatas"][0], s["distances"][0])

    gorulen = set()
    liste = []
    for metin, meta, mesafe in adaylar:
        if olumlu is not None and meta["olumlu"] != olumlu:
            continue
        if dil is not None and meta["dil"] != dil:
            continue
        if meta["review_id"] in gorulen:
            continue
        gorulen.add(meta["review_id"])
        liste.append({"mesafe": round(float(mesafe), 3), **meta, "metin": metin})
        if len(liste) == k:
            break
    return liste

testler = [
    ("Test 1: Dying Light, olumsuz, kasma", dict(soru="Oyun kasıyor mu, FPS düşüyor mu?", app_id=239140, olumlu=0)),
    ("Test 2: Dying Light, Türkçe",         dict(soru="Oyun kasıyor mu, FPS düşüyor mu?", app_id=239140, dil="tr", k=3)),
    ("Test 3: oyunsuz, Türkçe altyazı",     dict(soru="Oyunda Türkçe altyazı var mı?", k=3)),
]
for baslik, args in testler:
    t = time.time()
    sonuc = konu_ara(**args)
    sure = time.time() - t
    print(f"=== {baslik} | {len(sonuc)} sonuç | {sure:.2f} sn ===")
    yazdir(sonuc)

=== Test 1: Dying Light, olumsuz, kasma | 5 sonuç | 16.51 sn ===
0.404 | review_id=239140_111398 | parça 0/1 | olumlu=0 | dil=es | faydalı=0 | 2023-11-17
Si pudiera poner un "Meh", lo pondría.Es un juego repetitivo, pero eso no quita que es divertido, y la mecánica que lo separa de los demás juegos de zombies, el parkour, es bastante fluido aunque de vez en cuando te encontraras con una estructura que no puedes escalar porque al juego se le antoja no dejarte, o no detecta el input.Lo que más me causo problemas es el aspecto técnico del juego. Tengo una PC que supera por mucho los specs recomendados para jugar el juego y aún así le cuesta trabajo mantener los FPS a 60 o más y constantemente tienes bajones a los 20 e incluso a los 10 FPS sin importar la calidad gráfica o cuantas opciones desactives para quitar carga del GPU.Si lo encuentras en oferta lo puedo recomendar, pero no pagues precio completo por este juego ni sus DLCs
------------------------------------------------------------

## Adım 2c · konu_ara (hızlı sürüm)
- `oyun_parcalari(app_id)`: oyunun tüm parçaları (vektör + metin + metadata). Adım 3–4 de kullanacak.
- `konu_ara`: app_id varsa get + numpy, yoksa filtresiz HNSW (k×20 aday). olumlu/dil filtresi Python'da, review_id tekilleştirme. Yorumsuz oyunda boş liste.
- Sonuçlar Adım 2 ile birebir aynı ✓.
- Süre: Test 1 **16,5 sn** (Dying Light ilk çekiş, soğuk okuma), Test 2 0,29 sn (aynı oyun, önbellekte), Test 3 0,17 sn (HNSW).
  - Sorun: her yeni oyunun ilk çekişi yavaş olabilir → ölçülecek.
- Test 3 (oyunsuz, "Türkçe altyazı var mı?"): mesafe 0,21–0,23, 3/3 Türkçe dil desteği hakkında.
  - Türkçe dil beyanı yorumlardan doğrulanabilir (açık konu).
  - Oyunsuz aramada sonuçlar farklı oyunlardan → agent oyun adını belirtmeli.

In [5]:
soguk_oyunlar = {"Sims 4": 1222670, "BG3": 1086940, "Cyberpunk": 1091500}

for ad, app_id in soguk_oyunlar.items():
    t = time.time()
    sadece_id = yorumlar.get(where={"app_id": app_id}, include=[])
    t_id = time.time() - t

    t = time.time()
    tam = oyun_parcalari(app_id)
    t_tam1 = time.time() - t

    t = time.time()
    oyun_parcalari(app_id)
    t_tam2 = time.time() - t

    print(f"{ad:10} | parça: {len(sadece_id['ids']):4} | sadece id (soğuk): {t_id:6.2f} sn | "
          f"tam 1. kez: {t_tam1:6.2f} sn | tam 2. kez: {t_tam2:5.2f} sn")

Sims 4     | parça:  220 | sadece id (soğuk):  10.37 sn | tam 1. kez:   0.38 sn | tam 2. kez:  0.24 sn
BG3        | parça:  242 | sadece id (soğuk):   0.28 sn | tam 1. kez:   0.38 sn | tam 2. kez:  0.25 sn
Cyberpunk  | parça:  263 | sadece id (soğuk):   0.28 sn | tam 1. kez:   0.36 sn | tam 2. kez:  0.24 sn


## Adım 2d · Soğuk okuma testi
| Oyun | Parça | Sadece id (soğuk) | Tam, 1. kez | Tam, 2. kez |
|---|---|---|---|---|
| Sims 4 | 220 | **10,37 sn** | 0,38 sn | 0,24 sn |
| BG3 | 242 | 0,28 sn | 0,38 sn | 0,25 sn |
| Cyberpunk | 263 | 0,28 sn | 0,36 sn | 0,24 sn |

- Yavaşlık oyun başına değil, oturumdaki **ilk metadata filtresinde** (SQLite indeksi diskten RAM'e yükleniyor). Sonra her oyun ~0,3 sn.
- Adım 2c'deki 16,5 sn de bu: 15 dk boşta kalınca sayfalar RAM'den düşmüştü.
- Vektör + metin okumak sorun değil (0,36–0,38 sn).
- **Çözüm:** uygulama başlarken tek bir ısınma sorgusu (`get(where=app_id, include=[])`). 07 agent ve API başlangıcına eklenecek.
- Sonuç: oyunlu arama ~0,3 sn, oyunsuz arama ~0,2 sn (embedding 0,04 sn dahil).

In [6]:
def yorumlari_birlestir(h):
    yorumlar_sozluk = {}
    for metin, meta in zip(h["documents"], h["metadatas"]):
        rid = meta["review_id"]
        if rid not in yorumlar_sozluk:
            yorumlar_sozluk[rid] = {k: meta[k] for k in ["review_id", "app_id", "olumlu", "dil", "tarih", "faydali_oy"]}
            yorumlar_sozluk[rid]["parcalar"] = {}
        yorumlar_sozluk[rid]["parcalar"][meta["parca_no"]] = metin
    liste = []
    for y in yorumlar_sozluk.values():
        parcalar = y.pop("parcalar")
        y["metin"] = " [...] ".join(parcalar[i] for i in sorted(parcalar))
        liste.append(y)
    return liste

def ozetle(app_id, k=10, olumlu=None, dil=None):
    h = oyun_parcalari(app_id)
    liste = yorumlari_birlestir(h)
    if olumlu is not None:
        liste = [y for y in liste if y["olumlu"] == olumlu]
    if dil is not None:
        liste = [y for y in liste if y["dil"] == dil]
    liste.sort(key=lambda y: y["faydali_oy"], reverse=True)
    return liste[:k]

t = time.time()
hepsi = yorumlari_birlestir(oyun_parcalari(239140))
en_faydali = ozetle(239140, k=10)
print(f"süre: {time.time() - t:.2f} sn | Dying Light yorum sayısı (Chroma): {len(hepsi)}\n")

for y in en_faydali:
    print(f"faydalı={y['faydali_oy']:5} | olumlu={y['olumlu']} | dil={y['dil']} | "
          f"uzunluk={len(y['metin']):5} | {y['tarih']}")
    print(y["metin"][:300])
    print("-" * 80)

oran_hepsi = np.mean([y["olumlu"] for y in hepsi])
oran_top = np.mean([y["olumlu"] for y in en_faydali])
print(f"olumlu oran | Chroma'daki tüm DL yorumları: %{oran_hepsi * 100:.1f} | en faydalı 10: %{oran_top * 100:.1f}")

süre: 11.55 sn | Dying Light yorum sayısı (Chroma): 200

faydalı= 7142 | olumlu=1 | dil=zh | uzunluk=   76 | 2022-10-19
非常好的游戏，另外今天是只因节，点赞这段话，只因精灵就会在你睡着的时候偷偷给你的只因加长2厘米，但是如果你看到了这段话不点赞的话，你的只因就会缩短1厘米
--------------------------------------------------------------------------------
faydalı= 3463 | olumlu=1 | dil=zh | uzunluk=   76 | 2022-11-22
非常好的游戏，另外今天是牛子节，点赞这段话，牛子精灵就会在你睡着的时候偷偷给你的牛子加长2厘米，但是如果你看到了这段话不点赞的话，你的牛子就会缩短1厘米
--------------------------------------------------------------------------------
faydalı= 3287 | olumlu=1 | dil=zh | uzunluk=   67 | 2022-02-13
今天是牛子节，点赞这段话，牛子精灵就会在你睡着的时候偷偷给你的牛子加长1cm，但是如果你看到了这段话不点赞的话，你的牛子就会缩短1cm
--------------------------------------------------------------------------------
faydalı= 2928 | olumlu=0 | dil=zh | uzunluk=   33 | 2019-10-06
游戏中服装中国人的五星红旗为什么只有三颗星，强烈抵制，要求早点修改
--------------------------------------------------------------------------------
faydalı= 2638 | olumlu=1 | dil=zh | uzunluk=  253 | 2022-06-03
大概是今天下午5点左右，在哈兰监狱碰到一位ID叫    樱花不及你    的兄弟，他手上拿着BI

## Adım 3 · özetle (ilk hâli): faydalı oy tek başına çalışmıyor
- `yorumlari_birlestir`: parçalar review_id ile tam yoruma birleşir (" [...] " sınır işareti, 64 token örtüşme tekrarı kabul).
- `ozetle(app_id, k)`: tüm yorumlar → olumlu/dil filtresi → faydali_oy'a göre sırala.
- Dying Light (Chroma'da 200 yorum) en faydalı 10:
  - **10/10 Çince**, 7.142–2.015 faydalı oy.
  - 5'i zincir şakası ("beğenirsen ... 2 cm uzar"), 1'i oyunla ilgisiz bayrak şikâyeti, 2'si kişisel hikâye, 1'i espri.
  - Oyun hakkında bilgi: neredeyse yok (yarım cümle).
- Olumlu oran: tümü %95,0, en faydalı 10 %90,0 → sorun oran değil, içerik.
- **Neden:** Steam "faydalı" oyu "beğendim" gibi kullanılıyor. Çince topluluk şakalara binlerce oy veriyor, ham oy diller arasında karşılaştırılamıyor.
- Bellekteki "meme yorumlar faydalı oy alıyor" riski doğrulandı, beklenenden de ağır.
- Süre 11,5 sn = soğuk okuma (Adım 2d), ısınma sorgusuyla çözülecek.
- Çözüm adayları: (1) uzunluk ≥ 200, (2) önce "genel değerlendirme" sorgusuyla anlamsal seçim, sonra faydalı oy, (3) dil başına sıralama → Adım 3b'de karşılaştırılacak.

In [7]:
from collections import Counter

GENEL_SORGU = "Oyunun genel değerlendirmesi: oynanış, hikaye, grafikler, performans, artıları ve eksileri"

def ozetle_dene(app_id, yontem, k=8, aday=40):
    h = oyun_parcalari(app_id)
    liste = yorumlari_birlestir(h)

    if yontem == "uzunluk":
        liste = [y for y in liste if len(y["metin"]) >= 200]
        liste.sort(key=lambda y: y["faydali_oy"], reverse=True)
        return liste[:k]

    if yontem == "anlam":
        v = sorgu_vektoru(GENEL_SORGU)
        mesafeler = 1 - np.array(h["embeddings"]) @ v
        en_yakin = {}
        for meta, m in zip(h["metadatas"], mesafeler):
            rid = meta["review_id"]
            en_yakin[rid] = min(en_yakin.get(rid, 2.0), float(m))
        for y in liste:
            y["mesafe"] = round(en_yakin[y["review_id"]], 3)
        liste.sort(key=lambda y: y["mesafe"])
        secilen = liste[:aday]
        secilen.sort(key=lambda y: y["faydali_oy"], reverse=True)
        return secilen[:k]

    if yontem == "dil":
        gruplar = {}
        for y in sorted(liste, key=lambda y: y["faydali_oy"], reverse=True):
            gruplar.setdefault(y["dil"], []).append(y)
        diller = sorted(gruplar, key=lambda d: len(gruplar[d]), reverse=True)
        sonuc = []
        i = 0
        while len(sonuc) < k and i < max(len(g) for g in gruplar.values()):
            for d in diller:
                if i < len(gruplar[d]) and len(sonuc) < k:
                    sonuc.append(gruplar[d][i])
            i += 1
        return sonuc

hepsi = yorumlari_birlestir(oyun_parcalari(239140))
print("Dying Light yorumlarının dil dağılımı (Chroma):", Counter(y["dil"] for y in hepsi).most_common(), "\n")

for yontem in ["uzunluk", "anlam", "dil"]:
    print(f"========== YÖNTEM: {yontem} ==========")
    for y in ozetle_dene(239140, yontem):
        print(f"faydalı={y['faydali_oy']:5} | olumlu={y['olumlu']} | dil={y['dil']} | "
              f"uzunluk={len(y['metin']):5} | mesafe={y.get('mesafe', '-')}")
        print(y["metin"][:150])
        print("-" * 60)

Dying Light yorumlarının dil dağılımı (Chroma): [('zh', 77), ('en', 61), ('ru', 19), ('tr', 15), ('bilinmiyor', 8), ('pt', 8), ('es', 4), ('pl', 3), ('ko', 2), ('cs', 1), ('uk', 1), ('fr', 1)] 

========== YÖNTEM: uzunluk ==========
faydalı= 2638 | olumlu=1 | dil=zh | uzunluk=  253 | mesafe=-
大概是今天下午5点左右，在哈兰监狱碰到一位ID叫    樱花不及你    的兄弟，他手上拿着BIademaxx这把刀，我当时跟他说想要这把刀，他说他只有一把了，但还是给了我，我说帮他卡几把出来就还给他，然后不知道怎么回事房间的4个人都卡退了，我最后也没能把那把刀还给他，过后感觉很是惭愧，这位兄弟这么
------------------------------------------------------------
faydalı= 2404 | olumlu=1 | dil=zh | uzunluk=  670 | mesafe=-
刚开始玩这个游戏，也不怎么会玩，就是简单的做一做任务，然后卡在了”拿回第二个空投“这里，今天正在丧尸里面碰运气，结果进来一个玩家！哇，真的是活人哎。我想我会永远记住他的ID——Alone，他是我在逝光里认识的第一个人，第一个和我一起游戏的朋友了吧。他带我找到这个任务的那个空投箱子，带我做任务，一起打
------------------------------------------------------------
faydalı= 1985 | olumlu=1 | dil=zh | uzunluk=  736 | mesafe=-
近日steam出现了水军刷评论的情况，标题为给中国的友好建议，内容充斥着无知和反动，以及外国月亮圆的奴性思想，这些水军有一个共同特点，对留言做出评论，并且都是粘贴的内容，我想先问问那些水军，收了多少钱？？？BBC都敢吹？BBC的政治报道无不充斥着白皮猪的双标，同样一件事，自己的就到处洗，其他国家的就
---------

## Adım 3b · Üç özet yöntemi (Dying Light, 8'er yorum)
DL dil dağılımı (Chroma, 200 yorum): zh 77, en 61, ru 19, tr 15, bilinmiyor 8, pt 8, es 4, pl 3, ko 2, diğer 3.

| Yöntem | Nasıl | Oyunu değerlendiren | Sorun |
|---|---|---|---|
| uzunluk | ≥ 200 karakter → faydalı oy | 3/8 | Kişisel hikâye, siyasi tartışma, ipucu kaldı |
| **anlam** | GENEL_SORGU'ya en yakın 40 → faydalı oy | **8/8** | Hepsi olumlu |
| dil | dil başına faydalı sırası, sırayla | ~2/8 | Küçük dillerden tek satırlıklar, zh şaka geri geldi |

- "anlam" yönteminde mesafe 0,43–0,52. Diller doğal karıştı (5 en, 2 zh, 1 ru).
- Uzun ≠ ciddi. Faydalı oy ancak ilgili yorumlar arasında anlamlı.
- Eksik: olumsuz yorum hiç yok → olumlu/olumsuz ayrı gruplar önerisi. Oran SQL'den gelir.

In [8]:
def yorum_mesafeleri(h, soru):
    v = sorgu_vektoru(soru)
    mesafeler = 1 - np.array(h["embeddings"]) @ v
    en_yakin = {}
    for meta, m in zip(h["metadatas"], mesafeler):
        rid = meta["review_id"]
        en_yakin[rid] = min(en_yakin.get(rid, 2.0), float(m))
    return en_yakin

def ozetle(app_id, k_olumlu=6, k_olumsuz=3, dil=None, aday=40):
    h = oyun_parcalari(app_id)
    if len(h["ids"]) == 0:
        return {"yorum_sayisi": 0, "olumlu": [], "olumsuz": []}
    liste = yorumlari_birlestir(h)
    if dil is not None:
        liste = [y for y in liste if y["dil"] == dil]
    en_yakin = yorum_mesafeleri(h, GENEL_SORGU)
    for y in liste:
        y["mesafe"] = round(en_yakin[y["review_id"]], 3)

    sonuc = {"yorum_sayisi": len(liste)}
    for grup, deger, k in [("olumlu", 1, k_olumlu), ("olumsuz", 0, k_olumsuz)]:
        g = [y for y in liste if y["olumlu"] == deger]
        g.sort(key=lambda y: y["mesafe"])
        g = g[:aday]
        g.sort(key=lambda y: y["faydali_oy"], reverse=True)
        sonuc[grup + "_toplam"] = len([y for y in liste if y["olumlu"] == deger])
        sonuc[grup] = g[:k]
    return sonuc

for ad, app_id in [("Dying Light", 239140), ("Sims 4", 1222670)]:
    t = time.time()
    s = ozetle(app_id)
    print(f"########## {ad} | {time.time() - t:.2f} sn | Chroma'da {s['yorum_sayisi']} yorum "
          f"(olumlu {s['olumlu_toplam']}, olumsuz {s['olumsuz_toplam']}) ##########")
    for grup in ["olumlu", "olumsuz"]:
        print(f"--- {grup} ({len(s[grup])}) ---")
        for y in s[grup]:
            print(f"faydalı={y['faydali_oy']:5} | dil={y['dil']} | uzunluk={len(y['metin']):5} | mesafe={y['mesafe']}")
            print(y["metin"][:150])
            print("-" * 60)

########## Dying Light | 11.13 sn | Chroma'da 200 yorum (olumlu 190, olumsuz 10) ##########
--- olumlu (6) ---
faydalı= 1887 | dil=en | uzunluk= 5211 | mesafe=0.436
Glad to see the misinformed reviews have been thrown out of from the top page. For those who have said that Dying Light's day/night mechanic is not wh
------------------------------------------------------------
faydalı= 1455 | dil=en | uzunluk=  136 | mesafe=0.468
60 FPSNo CrashingVery pretty graphics.Not clunkyGuns actually work like guns (looking at you Dead Island)Zombies are satisfying to kill.
------------------------------------------------------------
faydalı= 1059 | dil=en | uzunluk= 1680 | mesafe=0.476
Lemme give it to y'all straight, who are on the fence;- Yes, the game is very much worth it at full price.- One of the first games in a long while, th
------------------------------------------------------------
faydalı= 1030 | dil=zh | uzunluk=  119 | mesafe=0.524
不错，这款游戏真的很好玩，可是有点太贵了，给男朋友口了一次才给我买，下次别给我推荐游戏了，真的，口腔溃

## Adım 3c · özetle (anlam + ayrı olumsuz grup)
- `yorum_mesafeleri`: yorum mesafesi = GENEL_SORGU'ya en yakın parçası.
- `ozetle(app_id, k_olumlu=6, k_olumsuz=3, dil, aday=40)` → {yorum_sayisi, olumlu, olumsuz, olumlu_toplam, olumsuz_toplam}. Genel oran SQL'den gelir.
- **Sims 4** (Chroma: 175 olumlu / 25 olumsuz): olumlular "eğlenceli, saatlerce oynanır", olumsuzlar ve olumluların yarısı **DLC fiyatı** (764 $, "oyunun %90'ı paralı DLC").
  - games.json %87 olumlu ile şikâyet dolu yorumların açıklaması bu: oyunu seviyorlar, fiyata kızıyorlar.
- **Dying Light** (190 / 10): olumlu 5/6 gerçek değerlendirme. Olumsuz 3'ün 2'si ilgisiz (bayrak, hile ifşası; mesafe 0,68).
  - Sebep: grup 10 yorum, aday=40 hiçbir şey elemiyor.
  - Çözüm: aday = grubun en yakın yarısı (en fazla 40).
- Sızanlar: müstehcen şaka (zh, 0,524), korsan DLC tavsiyesi (Sims 4) → agent kuralı: ilgisiz/şaka yorumları yok say, korsan yönlendirmesini aktarma.
- Hız: DL 11,1 sn (yine soğuk), Sims 4 0,44 sn. Birkaç dakikalık boşlukta bile soğuyor (RAM baskısı tahmini) → id eşlemesiyle çözülecek.

In [9]:
import math

def ozetle(app_id, k_olumlu=6, k_olumsuz=3, dil=None, aday=40):
    h = oyun_parcalari(app_id)
    if len(h["ids"]) == 0:
        return {"yorum_sayisi": 0, "olumlu": [], "olumsuz": []}
    liste = yorumlari_birlestir(h)
    if dil is not None:
        liste = [y for y in liste if y["dil"] == dil]
    en_yakin = yorum_mesafeleri(h, GENEL_SORGU)
    for y in liste:
        y["mesafe"] = round(en_yakin[y["review_id"]], 3)

    sonuc = {"yorum_sayisi": len(liste)}
    for grup, deger, k in [("olumlu", 1, k_olumlu), ("olumsuz", 0, k_olumsuz)]:
        g = [y for y in liste if y["olumlu"] == deger]
        sonuc[grup + "_toplam"] = len(g)
        n_aday = min(aday, max(k, math.ceil(len(g) / 2)))
        g.sort(key=lambda y: y["mesafe"])
        g = g[:n_aday]
        g.sort(key=lambda y: y["faydali_oy"], reverse=True)
        sonuc[grup] = g[:k]
    return sonuc

for ad, app_id in [("Dying Light", 239140), ("Sims 4", 1222670)]:
    t = time.time()
    s = ozetle(app_id)
    print(f"########## {ad} | {time.time() - t:.2f} sn | olumsuz toplam {s['olumsuz_toplam']} ##########")
    for y in s["olumsuz"]:
        print(f"faydalı={y['faydali_oy']:5} | dil={y['dil']} | uzunluk={len(y['metin']):5} | mesafe={y['mesafe']}")
        print(y["metin"][:150])
        print("-" * 60)

########## Dying Light | 16.70 sn | olumsuz toplam 10 ##########
faydalı=  575 | dil=zh | uzunluk= 1217 | mesafe=0.533
一个差评毁不了一款神作，从神作的标准给DL挑刺。    1.【夜魔】游戏里没有安排玩家去接触夜魔这个怪物（比如攻击方式和其血量，之前好像是让玩家躲着他。（但是后期我跟夜魔打，就发现夜魔基本无法用武器杀死（排除淹死和地形杀，另外我开的是噩梦难度）    2.【钩索】钩索勾不准又要钩多一次，这点很烦人，
------------------------------------------------------------
faydalı=    3 | dil=en | uzunluk=  330 | mesafe=0.503
Avrage game, with boring gameplay: respawn enemies, only few locatiaons which looks diffrent, etc. Nothing new in comparission with Dead Island, only 
------------------------------------------------------------
faydalı=    2 | dil=en | uzunluk=  472 | mesafe=0.543
Some of the worst melee combat I've encountered in a game and given that's the primary form of combat in this it's a pretty big downside. Most attacks
------------------------------------------------------------
########## Sims 4 | 10.19 sn | olumsuz toplam 25 ##########
faydalı= 3897 | dil=en | uzunluk=  437 | mesafe=0.597
90% of the game was cut

## Adım 3d · özetle: aday = grubun en yakın yarısı
- `n_aday = min(40, max(k, ceil(grup / 2)))`. DL olumsuz 10 → 5 aday, Sims olumsuz 25 → 13, büyük olumlu gruplar → 40 (değişmedi).
- **Dying Light olumsuz:** 3/3 gerçek eleştiri (gece yaratıkları ve kanca; tekrar eden oynanış; kötü yakın dövüş). Bayrak şikâyeti ve hile ifşası (0,68) elendi.
- **Sims 4 olumsuz:** 3/3 DLC fiyatı (0,575–0,64). Tek satırlık küfür (0,69) elendi.
- Faydalı oy artık ancak konuyla ilgili yorumlar arasında söz sahibi.
- **ÖZETLE KARARI:** GENEL_SORGU'ya mesafe → grubun en yakın yarısı (en fazla 40) → faydalı oy. Olumlu 6 + olumsuz 3 ayrı gruplar. Genel oran SQL'den gelir.
- Hız: DL 16,7 sn, Sims 4 10,2 sn (7 dk önce çekilmişti) → soğuk okuma birkaç dakikada geri geliyor. Çözüm: Adım 3e.

In [11]:
import os
import pandas as pd
from tqdm import tqdm

ESLEME_DOSYA = "../data/processed/chroma_parca_esleme.parquet"

if not os.path.exists(ESLEME_DOSYA):
    SAYFA = 10_000
    satirlar = []
    for bas in tqdm(range(0, yorumlar.count(), SAYFA)):
        s = yorumlar.get(limit=SAYFA, offset=bas, include=["metadatas"])
        for pid, m in zip(s["ids"], s["metadatas"]):
            satirlar.append((pid, m["app_id"]))
    pd.DataFrame(satirlar, columns=["parca_id", "app_id"]).to_parquet(ESLEME_DOSYA, index=False)

esleme = pd.read_parquet(ESLEME_DOSYA)
print(f"satır: {len(esleme):,} | parca_id tekil mi: {esleme['parca_id'].is_unique} | oyun: {esleme['app_id'].nunique():,}")

oyun_idleri = esleme.groupby("app_id")["parca_id"].apply(list).to_dict()

def oyun_parcalari(app_id):
    ids = oyun_idleri.get(app_id, [])
    if len(ids) == 0:
        return {"ids": [], "embeddings": [], "documents": [], "metadatas": []}
    return yorumlar.get(ids=ids, include=["embeddings", "documents", "metadatas"])

for ad, app_id in [("Elden Ring", 1245620), ("BG3", 1086940), ("Cyberpunk", 1091500)]:
    t = time.time()
    h = oyun_parcalari(app_id)
    t_yeni = time.time() - t
    t = time.time()
    eski = yorumlar.get(where={"app_id": app_id}, include=[])
    t_eski = time.time() - t
    print(f"{ad:10} | id ile: {t_yeni:5.2f} sn ({len(h['ids'])} parça) | "
          f"filtreyle: {t_eski:5.2f} sn ({len(eski['ids'])} parça) | aynı mı: {set(h['ids']) == set(eski['ids'])}")

100%|██████████| 265/265 [07:59<00:00,  1.81s/it]


satır: 2,644,490 | parca_id tekil mi: True | oyun: 22,898
Elden Ring | id ile:  0.25 sn (252 parça) | filtreyle:  2.59 sn (252 parça) | aynı mı: True
BG3        | id ile:  0.15 sn (242 parça) | filtreyle:  0.24 sn (242 parça) | aynı mı: True
Cyberpunk  | id ile:  0.18 sn (263 parça) | filtreyle:  0.25 sn (263 parça) | aynı mı: True


## Adım 3e · Parça id eşlemesi (soğuk okuma çözümü)
- `data/processed/chroma_parca_esleme.parquet`: parca_id → app_id. 2.644.490 satır, tekil ✓, 22.898 oyun (05'teki CSV'li oyun sayısı ✓).
- Üretim: `get(limit=10_000, offset=...)`, 265 sayfa, ~8 dk, tek seferlik.
  - 100.000'lik sayfa "too many SQL variables" hatası verdi (SQLite değişken sınırı ~32 bin).
  - Offset ilerledikçe sayfalar yavaşlıyor.
- `oyun_idleri`: {app_id: [parça id'leri]} sözlüğü, RAM'de.
- Yeni `oyun_parcalari`: `get(ids=...)`, yani birincil anahtar. konu_ara ve ozetle değişmeden kullanıyor.
- Test: 3 oyunda id ile = filtreyle (aynı parçalar ✓).
  - Elden Ring: id 0,25 sn, filtre 2,59 sn.
  - BG3 / Cyberpunk: ikisi de ~0,2 sn, çünkü tam tarama metadata'yı RAM'e yükledi (adil değil).
  - Soğukken hız sonraki hücrelerde izlenecek.

In [12]:
SAYIM_SORU = "Oyunda bug ve hata çok mu?"
app_id = 1091500  # Cyberpunk 2077

t = time.time()
h = oyun_parcalari(app_id)
liste = yorumlari_birlestir(h)
en_yakin = yorum_mesafeleri(h, SAYIM_SORU)
for y in liste:
    y["mesafe"] = round(en_yakin[y["review_id"]], 3)
liste.sort(key=lambda y: y["mesafe"])
print(f"süre: {time.time() - t:.2f} sn | yorum: {len(liste)}\n")

araliklar = [0, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 1.0]
mesafeler = np.array([y["mesafe"] for y in liste])
for alt, ust in zip(araliklar[:-1], araliklar[1:]):
    n = ((mesafeler >= alt) & (mesafeler < ust)).sum()
    print(f"{alt:.2f}–{ust:.2f}: {n:3} yorum")
print()

for sira, y in enumerate(liste[:30], 1):
    print(f"{sira:2}. {y['mesafe']} | olumlu={y['olumlu']} | dil={y['dil']} | {y['metin'][:120]}")

süre: 6.03 sn | yorum: 200

0.00–0.35:   4 yorum
0.35–0.40:  11 yorum
0.40–0.45:  22 yorum
0.45–0.50:  30 yorum
0.50–0.55:  36 yorum
0.55–0.60:  33 yorum
0.60–1.00:  64 yorum

 1. 0.243 | olumlu=1 | dil=tr | rtx4060 32 ram i7 13. nesil işlemci ile ara ara fps düşüşlerine tanık olmamın haricinde optimizasyon olarak bir hata ile
 2. 0.308 | olumlu=0 | dil=en | This game has so much potential....story was fantastic, the game was beautiful...only to be ruined by bugs. Even with th
 3. 0.309 | olumlu=0 | dil=zh | 你bug多也就算了，还有这么多影响主线的恶性bug，快把我恶心死了
 4. 0.347 | olumlu=1 | dil=en | Despite all the bugs, (i mean i don't think i found them all yet) the game has it's potential.The design itself it's the
 5. 0.352 | olumlu=0 | dil=en | Edit Jan 2022: I've put about a dozen hours into this on another platform after picking the game up for $20. The game is
 6. 0.354 | olumlu=1 | dil=en | I love this game but games needs to be optimized and fix glitches
 7. 0.367 | olumlu=1 | dil=tr | Oyunu bir kez %1

In [13]:
import json
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
llm = OpenAI()
ETIKET_MODEL = "gpt-4.1-mini"

ETIKET_TALIMAT = """Sana bir oyun hakkında bir soru ve bir oyuncu yorumu veriyorum.
Yorum bu soruya ne cevap veriyor?
- "evet": yorum sorudaki duruma evet diyor (ör. soru "bug çok mu?" ise yorum bug'lardan şikâyet ediyor)
- "hayir": yorum konuya değinip hayır diyor (ör. bug görmedim, düzeltildi, sorun yok)
- "ilgisiz": yorum bu konuya hiç değinmiyor
Sadece JSON döndür: {"etiket": "evet|hayir|ilgisiz", "gerekce": "Türkçe, en fazla 15 kelime"}"""

def etiketle(soru, metin):
    cevap = llm.chat.completions.create(
        model=ETIKET_MODEL, temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": ETIKET_TALIMAT},
                  {"role": "user", "content": f"Soru: {soru}\n\nYorum: {metin[:4000]}"}])
    return json.loads(cevap.choices[0].message.content)

def hepsini_getir(soru, app_id, esik=0.45, olumlu=None, dil=None):
    h = oyun_parcalari(app_id)
    if len(h["ids"]) == 0:
        return {"yorum_sayisi": 0, "aday": 0, "yorumlar": []}
    liste = yorumlari_birlestir(h)
    if olumlu is not None:
        liste = [y for y in liste if y["olumlu"] == olumlu]
    if dil is not None:
        liste = [y for y in liste if y["dil"] == dil]
    en_yakin = yorum_mesafeleri(h, soru)
    adaylar = []
    for y in liste:
        y["mesafe"] = round(en_yakin[y["review_id"]], 3)
        if y["mesafe"] <= esik:
            adaylar.append(y)
    with ThreadPoolExecutor(max_workers=8) as havuz:
        etiketler = list(havuz.map(lambda y: etiketle(soru, y["metin"]), adaylar))
    for y, e in zip(adaylar, etiketler):
        y["etiket"] = e.get("etiket")
        y["gerekce"] = e.get("gerekce")
    return {"yorum_sayisi": len(liste), "aday": len(adaylar), "yorumlar": adaylar}

t = time.time()
s = hepsini_getir("Oyunda bug ve hata çok mu?", 1091500)
print(f"süre: {time.time() - t:.1f} sn | örneklem: {s['yorum_sayisi']} yorum | aday (≤0,45): {s['aday']}\n")

df = pd.DataFrame(s["yorumlar"])
df["yil"] = df["tarih"].str[:4]
print(df["etiket"].value_counts(), "\n")
print(pd.crosstab(df["yil"], df["etiket"]), "\n")
for _, y in df.sort_values("mesafe").iterrows():
    print(f"{y['mesafe']} | {y['etiket']:8} | {y['yil']} | olumlu={y['olumlu']} | {y['gerekce']}")

süre: 5.9 sn | örneklem: 200 yorum | aday (≤0,45): 37

etiket
evet       24
hayir      11
ilgisiz     2
Name: count, dtype: int64 

etiket  evet  hayir  ilgisiz
yil                         
2020      13      2        2
2021       5      4        0
2022       2      1        0
2023       3      2        0
2024       1      2        0 

0.243 | evet     | 2023 | olumlu=1 | Yorumda oyunda çok sayıda bug ve hata olduğu belirtilmiş
0.308 | evet     | 2021 | olumlu=0 | Yorum oyunda çok bug olduğunu ve deneyimi bozduğunu söylüyor.
0.309 | evet     | 2020 | olumlu=0 | Yorum oyunda çok sayıda ve kötü bug olduğunu söylüyor
0.347 | evet     | 2021 | olumlu=1 | Yorumda çok sayıda bug olduğunu belirtiyor
0.352 | evet     | 2020 | olumlu=0 | Yorum oyunda çok sayıda bug ve çökme olduğunu belirtiyor
0.354 | evet     | 2020 | olumlu=1 | Yorum oyunda hatalar ve optimizasyon sorunlarından bahsediyor
0.367 | evet     | 2021 | olumlu=1 | Yorumda oyunda hala çok bug olduğu açıkça belirtilmiş.
0.37 | evet   

In [14]:
OYUN_TALIMAT = ("Instruct: Given a description of what a player is looking for, "
                "retrieve video games that match the description\nQuery: ")

def oyun_ara(istek, talimat=OYUN_TALIMAT, min_oy=None, k=5):
    v = sorgu_vektoru(istek, talimat=talimat)
    s = oyunlar.query(query_embeddings=[v.tolist()], n_results=k * 10,
                      include=["metadatas", "distances"])
    sonuc = []
    for meta, mesafe in zip(s["metadatas"][0], s["distances"][0]):
        if min_oy is not None and meta["oy_sayisi"] < min_oy:
            continue
        sonuc.append({"mesafe": round(mesafe, 3), **meta})
        if len(sonuc) == k:
            break
    return sonuc

istekler = [
    "uzayda geçen, gezegen keşfedilen açık dünya rol yapma oyunu",
    "arkadaşlarla birlikte oynanabilecek korku oyunu",
    "rahatlatıcı, stressiz bir çiftlik oyunu",
]
varyantlar = [
    ("A: yorum talimatı, filtresiz", dict(talimat=QWEN_TALIMAT)),
    ("B: oyun talimatı, filtresiz",  dict(talimat=OYUN_TALIMAT)),
    ("C: oyun talimatı, oy ≥ 100",   dict(talimat=OYUN_TALIMAT, min_oy=100)),
]

for istek in istekler:
    print(f"########## {istek} ##########")
    for ad, args in varyantlar:
        t = time.time()
        sonuc = oyun_ara(istek, **args)
        print(f"--- {ad} ({time.time() - t:.2f} sn)")
        for o in sonuc:
            print(f"  {o['mesafe']} | {o['ad'][:45]:45} | oy={o['oy_sayisi']:>7} | yetişkin={o['yetiskin']}")
    print()

########## uzayda geçen, gezegen keşfedilen açık dünya rol yapma oyunu ##########
--- A: yorum talimatı, filtresiz (6.47 sn)
  0.4 | Starfield                                     | oy= 160947 | yetişkin=0
  0.404 | helionaut                                     | oy=    201 | yetişkin=0
  0.43 | Sable                                         | oy=   5250 | yetişkin=0
  0.43 | Far Horizon                                   | oy=     30 | yetişkin=0
  0.439 | Lightyear Frontier                            | oy=   1813 | yetişkin=0
--- B: oyun talimatı, filtresiz (0.13 sn)
  0.334 | Starfield                                     | oy= 160947 | yetişkin=0
  0.354 | helionaut                                     | oy=    201 | yetişkin=0
  0.369 | Hazeron Starship                              | oy=     90 | yetişkin=0
  0.376 | Orange Cast: Sci-Fi Space Action Game         | oy=    158 | yetişkin=0
  0.377 | Journey To The Savage Planet                  | oy=   5792 | yetişkin=0
--- C: oyun talim

## Adım 5 · Oyun araması: talimat ve oy filtresi
- `oyun_ara(istek, talimat, min_oy, k)`: oyunlar koleksiyonunda HNSW (k×10 aday), oy filtresi Python'da.
- **Talimat:** OYUN_TALIMAT ("retrieve video games that match the description") 3/3 istekte mesafeyi düşürdü (Starfield 0,40 → 0,334), sıralama hafif iyileşti → **oyun araması OYUN_TALIMAT ile.**
- **Oy ≥ 100 yetersiz:** "arkadaşlarla korku" → INVITATION To FEAR (426), NIGHT CROW (100), SKIBIDI oyunları.
  - Phasmophobia / Lethal Company / Dead by Daylight yok.
  - Sebep: küçük oyunların açıklaması isteği kelimesi kelimesine tekrar ediyor. Embedding popülerliği bilmiyor.
- **Yetişkin sızıntısı:** çiftlik isteğinde HAREM！RANCH LIFE (yetiskin=1) → varsayılan olarak yetişkinleri hariç tut.
- İlk sorgu 6,5 sn (oyunlar indeksi yükleniyor), sonrakiler < 0,5 sn.

In [15]:
oyun_meta = oyunlar.get(include=["metadatas"])["metadatas"]
BILINENLER = ["Phasmophobia", "Lethal Company", "Dead by Daylight", "DEVOUR", "The Forest"]

print("--- Koleksiyonda var mı? ---")
for p in BILINENLER:
    bulunan = [(m["ad"], m["oy_sayisi"]) for m in oyun_meta if p.lower() in m["ad"].lower()]
    print(f"{p:18}: {bulunan[:3]}")

v = sorgu_vektoru("arkadaşlarla birlikte oynanabilecek korku oyunu", talimat=OYUN_TALIMAT)
ham = oyunlar.query(query_embeddings=[v.tolist()], n_results=200, include=["metadatas", "distances"])
print("\n--- Korku isteğinde ham sıra (ilk 200 içinde) ---")
for p in BILINENLER:
    sira = [(i + 1, round(d, 3)) for i, (m, d) in enumerate(zip(ham["metadatas"][0], ham["distances"][0]))
            if p.lower() in m["ad"].lower()]
    print(f"{p:18}: {sira[:1] if sira else 'ilk 200 içinde yok'}")

def oyun_ara(istek, w=0.02, min_oy=100, yetiskin_haric=True, k=5, aday=200, talimat=OYUN_TALIMAT):
    v = sorgu_vektoru(istek, talimat=talimat)
    s = oyunlar.query(query_embeddings=[v.tolist()], n_results=aday, include=["metadatas", "distances"])
    liste = []
    for meta, mesafe in zip(s["metadatas"][0], s["distances"][0]):
        if yetiskin_haric and meta["yetiskin"] == 1:
            continue
        if min_oy is not None and meta["oy_sayisi"] < min_oy:
            continue
        skor = mesafe - w * np.log10(meta["oy_sayisi"] + 1)
        liste.append({"skor": round(float(skor), 3), "mesafe": round(mesafe, 3), **meta})
    liste.sort(key=lambda o: o["skor"])
    return liste[:k]

print()
for istek in istekler:
    print(f"########## {istek} ##########")
    for w in [0, 0.02, 0.04, 0.06]:
        print(f"--- w = {w}")
        for o in oyun_ara(istek, w=w):
            print(f"  skor={o['skor']} | mesafe={o['mesafe']} | {o['ad'][:40]:40} | oy={o['oy_sayisi']:>7}")
    print()

--- Koleksiyonda var mı? ---
Phasmophobia      : [('Phasmophobia', 719060)]
Lethal Company    : [('Lethal Company', 443317)]
Dead by Daylight  : [('Hooked on You: A Dead by Daylight Dating Sim™', 5991)]
DEVOUR            : [('DEVOUR', 79359), ('Core Devourer', 97), ('The Devourer: Hunted Souls', 515)]
The Forest        : [('The House in the Forest', 41), ('The Forest Cathedral', 39), ('Sons Of The Forest', 209485)]

--- Korku isteğinde ham sıra (ilk 200 içinde) ---
Phasmophobia      : [(29, 0.358)]
Lethal Company    : ilk 200 içinde yok
Dead by Daylight  : ilk 200 içinde yok
DEVOUR            : [(111, 0.403)]
The Forest        : [(164, 0.429)]

########## uzayda geçen, gezegen keşfedilen açık dünya rol yapma oyunu ##########
--- w = 0
  skor=0.334 | mesafe=0.334 | Starfield                                | oy= 160947
  skor=0.354 | mesafe=0.354 | helionaut                                | oy=    201
  skor=0.376 | mesafe=0.376 | Orange Cast: Sci-Fi Space Action Game    | oy=    158
  s

## Adım 5b · Popülerlik bonusu + yetişkin filtresi
- `oyun_ara(istek, w, min_oy=100, yetiskin_haric=True, aday=200)`: skor = mesafe − w × log10(oy + 1), küçük skor üstte.
  - log10: 100 oy → 2, 100 bin → 5. 1.000 kat fark sadece 3 birim.
- **Koleksiyon kontrolü:**
  - Dead by Daylight veri setinde yok (sadece "Hooked on You").
  - Lethal Company var (443 bin oy) ama korku isteğinde ilk 200'de bile yok → etiket bazlı kriterler SQL'in işi.
  - Phasmophobia ham sırada 29., DEVOUR 111., Sons Of The Forest 164.
- **w = 0,02:** uzay en iyi (Starfield, Outer Wilds, Journey, Sable). Korkuda Phasmophobia 1., ama küçük oyunlar kalıyor.
- **w = 0,04:** korku en iyi (Phasmophobia, Escape the Backrooms, Labyrinthine, Content Warning). Uzayda kayma (Satisfactory, Horizon Zero Dawn).
- **w = 0,06:** popülerlik konuyu eziyor.
- Yetişkin oyunlar varsayılan olarak hariç (HAREM！RANCH LIFE artık çıkmıyor).
- w kesin değeri 08'de test sorularıyla ayarlanacak.

In [17]:
ZAYIF_ESIK = 0.45
METIN_SINIRI = 1200

def kisalt(metin, n=METIN_SINIRI):
    return metin if len(metin) <= n else metin[:n] + f" [kesildi, toplam {len(metin)} karakter]"

def yorum_satiri(y, n=METIN_SINIRI, isaretle=True):
    durum = "olumlu" if y["olumlu"] == 1 else "olumsuz"
    zayif = " ZAYIF EŞLEŞME" if isaretle and y.get("mesafe", 0) > ZAYIF_ESIK else ""
    mesafe = f" | mesafe={y.get('mesafe', '-')}{zayif}" if isaretle else ""
    return (f"[app_id={y['app_id']} | {durum} | dil={y['dil']} | {y['tarih']} | "
            f"faydalı={y['faydali_oy']}{mesafe}]\n{kisalt(y['metin'], n)}")

def yorum_araci(mod, soru=None, app_id=None, olumlu=None, dil=None):
    if mod == "konu":
        if not soru:
            return "HATA: 'konu' modu için soru gerekli."
        liste = konu_ara(soru, app_id=app_id, olumlu=olumlu, dil=dil, k=5)
        if not liste:
            return "Bu oyun için yorum verisi yok." if app_id else "Sonuç bulunamadı."
        bas = f"KONU ARAMASI: {len(liste)} yorum, en ilgiliden aşağı. Mesafe {ZAYIF_ESIK} üstü = zayıf eşleşme, kanıt sayma."
        return bas + "\n\n" + "\n\n".join(yorum_satiri(y) for y in liste)

    if mod == "ozetle":
        if app_id is None:
            return "HATA: 'ozetle' modu için app_id gerekli. Önce oyun_bul ile oyunu bul."
        s = ozetle(app_id, dil=dil)
        if s["yorum_sayisi"] == 0:
            return "Bu oyun için yorum verisi yok. Sadece SQL'deki sayıları kullan ve bunu kullanıcıya söyle."
        bas = (f"ÖZET: örneklemde {s['yorum_sayisi']} yorum var (olumlu {s['olumlu_toplam']}, olumsuz {s['olumsuz_toplam']}). "
               f"Bu oran genel puan değildir, genel puanı SQL'den al. Aşağıdakiler oyunu değerlendiren en faydalı yorumlar.")
        olumlu_k = "\n\n".join(yorum_satiri(y, isaretle=False) for y in s["olumlu"])
        olumsuz_k = "\n\n".join(yorum_satiri(y, isaretle=False) for y in s["olumsuz"]) or "(olumsuz yorum yok)"
        return f"{bas}\n\n=== OLUMLU ===\n{olumlu_k}\n\n=== OLUMSUZ ===\n{olumsuz_k}"

    if mod == "hepsi":
        if not soru or app_id is None:
            return "HATA: 'hepsi' modu için soru ve app_id gerekli."
        s = hepsini_getir(soru, app_id, olumlu=olumlu, dil=dil)
        if s["yorum_sayisi"] == 0:
            return "Bu oyun için yorum verisi yok."
        if s["aday"] == 0:
            return f"Örneklemdeki {s['yorum_sayisi']} yorumdan hiçbiri bu konuya değinmiyor."
        df = pd.DataFrame(s["yorumlar"])
        df["yil"] = df["tarih"].str[:4]
        sayim = df["etiket"].value_counts().to_dict()
        yillar = pd.crosstab(df["yil"], df["etiket"]).to_csv()
        bas = (f"SAYIM: örneklemdeki {s['yorum_sayisi']} yorumun {s['aday']}'i konuya yakın. "
               f"LLM etiketi: {sayim}. evet = soruya evet diyor, hayir = konuya değinip hayır diyor. "
               f"Sayılar örneklemden, Steam'in tamamı için mutlak sayı söyleme. Kaçırılan yorum olabilir, oranı 'en az' diye oku.")
        ornekler = []
        for etiket in ["evet", "hayir"]:
            for _, y in df[df["etiket"] == etiket].sort_values("mesafe").head(3).iterrows():
                ornekler.append(f"({etiket}: {y['gerekce']})\n" + yorum_satiri(y, n=300))
        return f"{bas}\n\nYıllara göre:\n{yillar}\nÖrnekler:\n\n" + "\n\n".join(ornekler)

    return f"HATA: bilinmeyen mod '{mod}'. Geçerli modlar: konu, ozetle, hepsi."

def oyun_araci(istek, yetiskin_haric=True):
    liste = oyun_ara(istek, w=0.03, yetiskin_haric=yetiskin_haric, k=10)
    df = pd.DataFrame(liste)[["app_id", "ad", "oy_sayisi", "mesafe"]]
    return ("OYUN ARAMASI (tarife göre benzerlik + popülerlik, en uygundan aşağı). "
            "Puan/fiyat/etiket gibi kesin bilgileri SQL'den kontrol et.\n" + df.to_csv(index=False))

print(yorum_araci("ozetle", app_id=1222670)[:1500])

ÖZET: örneklemde 200 yorum var (olumlu 175, olumsuz 25). Bu oran genel puan değildir, genel puanı SQL'den al. Aşağıdakiler oyunu değerlendiren en faydalı yorumlar.

=== OLUMLU ===
[app_id=1222670 | olumlu | dil=en | 2024-06-30 | faydalı=1488]
best game ever. if you want all DLCs just search up Anadius and enjoy for free.

[app_id=1222670 | olumlu | dil=en | 2022-10-09 | faydalı=1321]
Only People like Elon Musk can buy the whole game! my advise to you is Just to buy the base game and install mods instead.I used mods (Wicked Williams) to turn it into the ULTIMATE SEX game!

[app_id=1222670 | olumlu | dil=zh | 2020-12-19 | faydalı=1181]
我对这个游戏真的是又爱又恨，这游戏又让人入迷，但是定价却让人望而止步，除了新的几个dlc我大部分都买了，我还在在打折的情况下花了一千多（就离谱 )，但不得不说游戏是真好玩，真的把现实生活体现出来了。后面就说一下我个人觉得必买的dlc玩转大学（可以去上大学，然后上课，挺有意思的，特别是把小孩子养大送去大学很有感觉）来去上班（上班996，有很多职业可以做的，推荐游戏主播不错，很赚钱，还有像医生、警察的职业可以亲自去上班，然后根据工作升职，还可以自己创业开店）星梦起飞（当明星，我之前到了五星，出个门一堆人围着我拍照要签名）猫狗总动员（养宠物，各种猫和狗）都会生活（有个城市的图，可以搬去城市住）同欢共乐（可以开party开俱乐部）春夏秋冬（季节更替更有代入感）生儿育女（就是可以生孩子带孩子，不建议养一个以上的小孩，

## Adım 6 · Agent'ın göreceği araç çıktıları
- `yorum_araci(mod, soru, app_id, olumlu, dil)` → metin. Modlar: konu / ozetle / hepsi. Eksik parametrede `HATA: ...` + ne yapılacağı.
- `oyun_araci(istek, yetiskin_haric=True)` → CSV (app_id, ad, oy_sayisi, mesafe), w = 0,03, 10 oyun.
- Her yorumun künyesi: app_id | olumlu/olumsuz | dil | tarih | faydalı. Konu ve hepsi modlarında + mesafe. Metin 1.200 karakterde kesilir.
- Çıktıya gömülü uyarılar:
  - zayıf eşleşme (mesafe > 0,45, sadece konu modunda)
  - "bu oran genel puan değil, SQL'den al" (özet)
  - "sayılar örneklemden, en az diye oku" (hepsi)
  - "yorum verisi yok" (CSV'siz oyun, ör. Starfield)

| Çıktı | Token (≈) | Süre |
|---|---|---|
| konu (Dying Light) | 650 | 5,4 sn (soğuk) |
| ozetle (Sims 4) | ~1.100 | 0,2 sn |
| hepsi (Cyberpunk) | 785 | 4,9 sn (LLM dahil) |
| yok (Starfield) | 26 | 0,0 sn |
| oyun araması | 150–160 | 0,2 sn |

- **Düzeltme:** özetteki mesafe kullanıcı sorusuna değil GENEL_SORGU'ya göre (0,54–0,60), bu yüzden hepsi "zayıf" görünüyordu → özette mesafe ve işaret kaldırıldı (`yorum_satiri(..., isaretle=False)`).
- 4b'de şüpheli sanılan Türkçe etiket doğruymuş: FPS sorunu yok diyor, ama "oyun hala bug dolu".
- Etiketler çalıştırmalar arasında ±1 oynuyor (hayır 11 → 10, ilgisiz 2 → 3), temperature 0 olsa da → 08'de not.
- w = 0,03 doğrulandı: uzayda ilk 4 doğru, korkuda ilk 4 popüler co-op korku (Phasmophobia, Escape the Backrooms, Labyrinthine, Content Warning), çiftlik temiz.
- Kalan riskler, sayıyla elenemiyor → 6b kurallarında: ironik şaka yorumlar (konu aramasında 1. sırada), korsan/mod tavsiyesi (Sims 4 özetinde).

In [18]:
RAG_ARAC_ACIKLAMASI = """Oyuncu yorumlarında anlamsal arama (Chroma). Yorum metinleri burada, sayılar SQL'de.
Önce oyun_bul ile oyunun app_id'sini bul.

Modlar:
- konu: belirli bir konudaki en ilgili 5 yorum ("gece korkutucu mu?", "kasma var mı?"). app_id verilmezse tüm oyunlarda arar.
- ozetle: oyun hakkında genel görüş. Oyunu değerlendiren en faydalı 6 olumlu + 3 olumsuz yorum. app_id şart.
- hepsi: bir konuda yorumları sayar ("kaç yorum bug'dan şikâyet ediyor?"). Konuya yakın tüm yorumlar LLM ile evet/hayır/ilgisiz etiketlenir, yıllara göre tablo döner. soru + app_id şart. Yavaş (~5 sn).

Kurallar:
1. Genel puan ve olumlu oranı SQL'den (oyunlar.olumlu_oran). Bu araç "neden" sorusu için. Oran ile yorumlar çelişirse açıkla (ör. Sims 4: oyunu seviyorlar, DLC fiyatına kızıyorlar).
2. Yorumlar örneklemdir (oyun başına en fazla 200). Sayıları "örneklemdeki N yorumun X'i" diye ver, Steam'in tamamı için mutlak sayı söyleme.
3. "ZAYIF EŞLEŞME" yazan yorumu kanıt sayma. Hepsi zayıfsa "yorumlarda bu konuda net bilgi yok" de.
4. Faydalı oy ciddiyet garantisi değildir. İronik, şaka, zincir mesaj veya oyunla ilgisiz yorumları yok say (ör. "korku yok, çocuklar için uygun" diyen ironik yorum).
5. Korsan indirme, hile veya cinsel içerikli mod tavsiyelerini kullanıcıya aktarma.
6. Tarihe dikkat et: eski yorumlar yamalardan önceki durumu anlatabilir. "Düzeldi mi?" sorularında yıllara bak. Yorumlar sonradan düzenlenebilir ("Edit ...").
7. Yorumlar farklı dillerde olabilir. Türkçe özetle, alıntıyı kısa tut.
8. "Yorum verisi yok" dönerse bunu kullanıcıya söyle, sadece SQL'deki bilgilerle cevap ver.
9. Türkçe dil desteği sorularında SQL'deki beyanı (oyun_dil) yorumlarla doğrula (konu modu, soru "Türkçe dil desteği var mı").
"""

OYUN_ARAC_ACIKLAMASI = """Tarife göre oyun önerir (anlamsal benzerlik + popülerlik). 10 oyun döner: app_id, ad, oy_sayisi, mesafe.
Bulanık istekler için: "uzayda geçen keşif oyunu", "rahatlatıcı çiftlik oyunu".
Kurallar:
1. Kesin kriterler (etiket, tür, co-op, dil desteği, fiyat, puan eşiği) SQL'in işi. Bu aracın sonuçlarını önermeden önce SQL ile puan/etiket kontrol et.
2. Bazı ünlü oyunlar listede çıkmayabilir (açıklaması tarife benzemiyorsa) veya veri setinde hiç olmayabilir. Gerekirse SQL'de etiketle de ara.
3. Yetişkin oyunlar varsayılan olarak hariç. Sadece kullanıcı açıkça yetişkin içerik isterse yetiskin_haric=false ver. Çocuk/aile sorularında asla false verme.
"""

RAG_ARAC_TANIMLARI = [
    {"type": "function", "function": {
        "name": "yorum_ara",
        "description": RAG_ARAC_ACIKLAMASI,
        "parameters": {"type": "object", "properties": {
            "mod":    {"type": "string", "enum": ["konu", "ozetle", "hepsi"]},
            "soru":   {"type": "string", "description": "Kullanıcının sorusu, oyun adı olmadan. konu ve hepsi modunda şart."},
            "app_id": {"type": "integer", "description": "oyun_bul'dan gelen app_id. ozetle ve hepsi modunda şart."},
            "olumlu": {"type": "integer", "enum": [0, 1], "description": "Sadece olumsuz (0) veya olumlu (1) yorumlar."},
            "dil":    {"type": "string", "description": "Yorum dili kodu: tr, en, zh, ru, de, es, pt, fr..."}},
            "required": ["mod"]}}},
    {"type": "function", "function": {
        "name": "oyun_ara",
        "description": OYUN_ARAC_ACIKLAMASI,
        "parameters": {"type": "object", "properties": {
            "istek":          {"type": "string", "description": "Kullanıcının aradığı oyunun tarifi."},
            "yetiskin_haric": {"type": "boolean", "description": "Varsayılan true."}},
            "required": ["istek"]}}},
]

RAG_FONKSIYONLAR = {"yorum_ara": yorum_araci, "oyun_ara": oyun_araci}

def rag_arac_cagir(ad, argumanlar_json):
    if ad not in RAG_FONKSIYONLAR:
        return f"HATA: bilinmeyen araç '{ad}'. Geçerli: {list(RAG_FONKSIYONLAR)}."
    try:
        argumanlar = json.loads(argumanlar_json) if argumanlar_json else {}
    except json.JSONDecodeError:
        return "HATA: argümanlar geçerli JSON değil."
    for alan in ["app_id", "olumlu"]:
        if alan in argumanlar and isinstance(argumanlar[alan], str) and argumanlar[alan].isdigit():
            argumanlar[alan] = int(argumanlar[alan])
    try:
        return RAG_FONKSIYONLAR[ad](**argumanlar)
    except TypeError as e:
        return f"HATA: yanlış parametre ({e}). Şemadaki parametre adlarını kullan."
    except Exception as e:
        return f"HATA: araç çalışırken sorun çıktı ({type(e).__name__}: {e})."

print(f"Tanımların boyutu: {len(json.dumps(RAG_ARAC_TANIMLARI, ensure_ascii=False)):,} karakter "
      f"≈ {len(json.dumps(RAG_ARAC_TANIMLARI, ensure_ascii=False)) // 3.4:,.0f} token\n")

denemeler = [
    ("doğru, app_id metin olarak", "yorum_ara", '{"mod": "konu", "soru": "Gece korkutucu mu?", "app_id": "239140"}'),
    ("bozuk JSON",                  "yorum_ara", '{"mod": "konu", "soru": '),
    ("yanlış parametre adı",        "yorum_ara", '{"mod": "konu", "sorgu": "kasma var mı"}'),
    ("eksik app_id",                "yorum_ara", '{"mod": "ozetle"}'),
    ("bilinmeyen mod",              "yorum_ara", '{"mod": "say", "app_id": 239140}'),
    ("bilinmeyen araç",             "yorum_bul", '{"mod": "konu"}'),
    ("oyun araması",                "oyun_ara",  '{"istek": "çocuğumla oynayabileceğim tatlı bir platform oyunu"}'),
]
for baslik, ad, arg in denemeler:
    cikti = rag_arac_cagir(ad, arg)
    print(f"--- {baslik}\n{cikti[:400]}\n")

Tanımların boyutu: 3,354 karakter ≈ 986 token

--- doğru, app_id metin olarak
KONU ARAMASI: 5 yorum, en ilgiliden aşağı. Mesafe 0.45 üstü = zayıf eşleşme, kanıt sayma.

[app_id=239140 | olumlu | dil=en | 2015-01-31 | faydalı=684 | mesafe=0.324]
"The Walking Dead" by day, "I Am Legend" by night;I can't tell you how beautiful this game looks without you just seeing it for yourself,  but I will say this.This is the first game, in a very long time, which has legitamately inspir

--- bozuk JSON
HATA: argümanlar geçerli JSON değil.

--- yanlış parametre adı
HATA: yanlış parametre (yorum_araci() got an unexpected keyword argument 'sorgu'). Şemadaki parametre adlarını kullan.

--- eksik app_id
HATA: 'ozetle' modu için app_id gerekli. Önce oyun_bul ile oyunu bul.

--- bilinmeyen mod
HATA: bilinmeyen mod 'say'. Geçerli modlar: konu, ozetle, hepsi.

--- bilinmeyen araç
HATA: bilinmeyen araç 'yorum_bul'. Geçerli: ['yorum_ara', 'oyun_ara'].

--- oyun araması
OYUN ARAMASI (tarife göre benzerlik + po

## Adım 6b · Araç tanımları, kurallar ve rag_arac_cagir
- `RAG_ARAC_TANIMLARI`: iki araç, OpenAI JSON şema (04 ile aynı biçim).
  - `yorum_ara`: mod (konu/ozetle/hepsi), soru, app_id, olumlu (0/1), dil.
  - `oyun_ara`: istek, yetiskin_haric.
- Boyut: 3.354 karakter ≈ 986 token (SQL tanımlarıyla birlikte ~2.400).
- Açıklamadaki kurallar = 06'nın bulguları: oran SQL'den / çelişkiyi açıkla, örneklem sayısı, ZAYIF kanıt değil, ironik/şaka yorumları yok say, korsan/hile/cinsel mod tavsiyesi aktarma, tarih/yama, Türkçe özetle, yorum yok → söyle, Türkçe beyanı yorumla doğrula. Oyun aramasında kesin kriter SQL'de, ünlü oyun eksik olabilir, yetişkin varsayılan hariç.
- `rag_arac_cagir(ad, argumanlar_json)`: 7/7 test doğru. Metin app_id sayıya çevriliyor, hatalar `HATA: ...` + ipucu olarak dönüyor.
- "Çocuğumla platform oyunu" → Webbed, Fireboy & Watergirl, Bread & Fred, Bluey, Tinykin (yetişkin filtresi ✓).